# LeetCode 1283 — Find the Smallest Divisor Given a Threshold bằng Python

Link đề: https://leetcode.com/problems/find-the-smallest-divisor-given-a-threshold/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def smallestDivisor(self, nums: list[int], threshold: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu và cài đặt chính xác **phép chia làm tròn lên** (ceil) bằng số nguyên: `(a + d - 1) // d` hoặc `-(-a // d)`.
2. Nhận ra **tính đơn điệu**: số chia càng lớn thì tổng càng nhỏ ⇒ có thể **binary search trên đáp án**.
3. Viết mẫu binary search "**tìm giá trị nhỏ nhất thoả điều kiện**" (`lo < hi`, `hi = mid` / `lo = mid + 1`) không bị lặp vô hạn, không lệch 1.
4. Biết thu hẹp khoảng tìm kiếm bằng **cận dưới `ceil(sum/threshold)`** và tăng tốc tính tổng bằng **đếm tần suất** (nâng cao).
5. Nhận dạng họ bài "binary search on answer": LC 875, 1011, 410, 1482, 2187…

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums` và số nguyên `threshold`. Chọn một số nguyên dương `divisor`, chia **từng phần tử** cho `divisor` rồi **làm tròn lên** (ví dụ `7/3 → 3`, `10/2 → 5`), sau đó cộng các kết quả. Tìm **`divisor` nhỏ nhất** sao cho tổng này `≤ threshold`. Đề đảm bảo luôn có đáp án.

**Ví dụ 1:** `nums = [1,2,5,9], threshold = 6` → `5`.
- `divisor = 1`: tổng `1+2+5+9 = 17`;
- `divisor = 4`: tổng `1+1+2+3 = 7`;
- `divisor = 5`: tổng `1+1+1+2 = 5 ≤ 6` ✔.

**Ví dụ 2:** `nums = [44,22,33,11,1], threshold = 5` → `44`.

**Ràng buộc:** `1 <= nums.length <= 5·10^4`, `1 <= nums[i] <= 10^6`, `nums.length <= threshold <= 10^6`.

In [1]:
import bisect
import math
import random
import time
from typing import List

examples = [([1, 2, 5, 9], 6, 5), ([44, 22, 33, 11, 1], 5, 44)]
for nums, th, want in examples:
    print(f"nums={nums}, threshold={th} -> mong đợi {want}")

nums=[1, 2, 5, 9], threshold=6 -> mong đợi 5
nums=[44, 22, 33, 11, 1], threshold=5 -> mong đợi 44


## 3. Kiến thức nền

### 3.1 Chia làm tròn lên (ceil) bằng số nguyên

- `math.ceil(a / d)`: dùng số thực, đúng với số nhỏ nhưng có thể sai với số rất lớn (sai số float).
- `(a + d - 1) // d`: an toàn, chỉ dùng số nguyên (với `a ≥ 0`, `d > 0`).
- `-(-a // d)`: mẹo gọn — `//` làm tròn xuống, đổi dấu hai lần thành làm tròn lên.

In [2]:
for a, d in [(7, 3), (10, 2), (1, 5), (9, 5), (0, 4)]:
    print(f"{a}/{d}: ceil={math.ceil(a / d)}, (a+d-1)//d={(a + d - 1) // d}, -(-a//d)={-(-a // d)}")

big = 10**17 + 1
print("float sai với số lớn:", math.ceil(big / 1), "vs", (big + 1 - 1) // 1)

7/3: ceil=3, (a+d-1)//d=3, -(-a//d)=3
10/2: ceil=5, (a+d-1)//d=5, -(-a//d)=5
1/5: ceil=1, (a+d-1)//d=1, -(-a//d)=1
9/5: ceil=2, (a+d-1)//d=2, -(-a//d)=2
0/4: ceil=0, (a+d-1)//d=0, -(-a//d)=0
float sai với số lớn: 100000000000000000 vs 100000000000000001


### 3.2 Hàm tổng `f(d)` và tính đơn điệu

Định nghĩa `f(d) = Σ ceil(nums[i] / d)`. Khi `d` tăng, mỗi số hạng **không tăng** ⇒ `f` **không tăng**. Hàm đơn điệu ⇒ tập các `d` thoả `f(d) ≤ threshold` là một **đoạn liên tục đến vô cùng** `[ans, ∞)`.

In [3]:
def f(nums, d):
    return sum((x + d - 1) // d for x in nums)

nums = [1, 2, 5, 9]
print([(d, f(nums, d)) for d in range(1, 11)])

[(1, 17), (2, 10), (3, 7), (4, 7), (5, 5), (6, 5), (7, 5), (8, 5), (9, 4), (10, 4)]


### 3.3 Binary search "tìm vị trí đầu tiên thoả"

Cho vị từ `ok(d)` có dạng `False, False, …, False, True, True, …`. Tìm `d` nhỏ nhất có `ok(d) = True` trong `[lo, hi]`:

```python
while lo < hi:
    mid = (lo + hi) // 2
    if ok(mid): hi = mid        # mid có thể là đáp án, giữ lại
    else:       lo = mid + 1    # mid chắc chắn không phải
return lo
```

Mỗi vòng khoảng `[lo, hi]` co lại ít nhất 1, và đáp án luôn nằm trong khoảng.

In [4]:
def first_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(first_true(0, 100, lambda x: x * x >= 50))   # 8
print(first_true(1, 10, lambda d: f(nums, d) <= 6))

8
5


### 3.4 `bisect` với tham số `key`

Từ Python 3.10: `bisect.bisect_left(range(lo, hi + 1), True, key=ok)` tìm vị trí đầu tiên mà `ok` cho `True` (vì `False < True`). Không cần tạo mảng thật — `range` hỗ trợ truy cập chỉ số `O(1)`.

In [5]:
r = range(1, 11)
i = bisect.bisect_left(r, True, key=lambda d: f(nums, d) <= 6)
print("chỉ số:", i, "-> divisor =", r[i])

chỉ số: 4 -> divisor = 5


### 3.5 Đếm tần suất và prefix count

`cnt[v]` = số phần tử bằng `v`; `pre[v]` = số phần tử `≤ v`. Khi đó số phần tử trong khoảng `(a, b]` là `pre[b] − pre[a]` — dùng ở Cách 5 để tính `f(d)` nhanh khi `d` lớn.

In [6]:
arr = [1, 2, 2, 5, 9]
M = max(arr)
cnt = [0] * (M + 1)
for x in arr:
    cnt[x] += 1
pre = cnt[:]
for v in range(1, M + 1):
    pre[v] += pre[v - 1]
print("cnt =", cnt)
print("pre =", pre, "| số phần tử trong (2, 9] =", pre[9] - pre[2])

cnt = [0, 1, 2, 0, 0, 1, 0, 0, 0, 1]
pre = [0, 1, 3, 3, 3, 4, 4, 4, 4, 5] | số phần tử trong (2, 9] = 2


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** `nums = [1, 2, 5, 9]`, `threshold = 6`:

| d | ceil(1/d) | ceil(2/d) | ceil(5/d) | ceil(9/d) | f(d) | ≤ 6? |
|---|---|---|---|---|---|---|
| 1 | 1 | 2 | 5 | 9 | 17 | ✗ |
| 2 | 1 | 1 | 3 | 5 | 10 | ✗ |
| 3 | 1 | 1 | 2 | 3 | 7 | ✗ |
| 4 | 1 | 1 | 2 | 3 | 7 | ✗ |
| 5 | 1 | 1 | 1 | 2 | 5 | ✔ |
| 9 | 1 | 1 | 1 | 1 | 4 | ✔ |

Hai quan sát:
1. `f(d)` giảm dần (không tăng) ⇒ cột "≤ 6?" có dạng ✗✗✗✗✔✔… ⇒ tìm điểm chuyển đầu tiên.
2. Với `d = max(nums)`, mọi số hạng bằng 1 ⇒ `f = n ≤ threshold` (ràng buộc cho `n ≤ threshold`) ⇒ đáp án luôn `≤ max(nums)`.

In [7]:
for nums, th, want in examples:
    table = [(d, f(nums, d)) for d in range(1, max(nums) + 1)]
    first_ok = next(d for d, s in table if s <= th)
    print(f"threshold={th}: f(1)={table[0][1]}, f(max)={table[-1][1]}, d đầu tiên thoả = {first_ok}")

threshold=6: f(1)=17, f(max)=4, d đầu tiên thoả = 5
threshold=5: f(1)=111, f(max)=5, d đầu tiên thoả = 44


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử `d = 1, 2, 3, …` cho tới khi `f(d) ≤ threshold`. Theo quan sát 2, chắc chắn dừng trước hoặc tại `max(nums)`.

In [8]:
def smallest_divisor_brute(nums, threshold):
    for d in range(1, max(nums) + 1):
        if sum((x + d - 1) // d for x in nums) <= threshold:
            return d
    return max(nums)          # không bao giờ tới đây nếu n <= threshold

for nums, th, want in examples:
    print(smallest_divisor_brute(nums, th), "| mong đợi", want)

5 | mong đợi 5
44 | mong đợi 44


**Độ phức tạp:** mỗi lần thử tốn `O(n)`, thử tới `M = max(nums)` lần ⇒ `O(n·M)`. Worst: `n = 5·10^4`, `M = 10^6`, đáp án lớn ⇒ `5·10^10` phép — không khả thi.

**Nút thắt:** thử tuần tự từng `d`, bỏ qua tính đơn điệu. Biết `f` đơn điệu thì mỗi lần thử có thể **loại một nửa** khoảng.

In [9]:
# Thấy nút thắt: đáp án lớn => brute force phải thử rất nhiều d
nums_bad = [10**5] * 200
th_bad = 200                       # buộc d = 10^5
t = time.perf_counter()
print(smallest_divisor_brute(nums_bad, th_bad), f"{time.perf_counter() - t:.2f}s cho n=200, M=10^5")

100000 1.36s cho n=200, M=10^5


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** `ok(d) := f(d) ≤ threshold` là vị từ **đơn điệu** (một khi đúng thì đúng mãi khi `d` tăng). Đáp án là `d` nhỏ nhất có `ok(d)`.

**Khoảng tìm kiếm:** `lo = 1`, `hi = max(nums)` (vì `ok(max) = True`).

**Quy tắc "nếu… thì…":**
- Nếu `ok(mid)` ⇒ đáp án `≤ mid` ⇒ `hi = mid`.
- Nếu không ⇒ đáp án `> mid` ⇒ `lo = mid + 1`.
- Khi `lo == hi` ⇒ đó là đáp án.

**Invariant:** đáp án luôn nằm trong `[lo, hi]`; `ok(hi)` luôn đúng; mọi `d < lo` đều có `ok(d)` sai.

**Cải tiến cận dưới:** vì `ceil(x/d) ≥ x/d`, ta có `f(d) ≥ S/d` với `S = sum(nums)`. Muốn `f(d) ≤ threshold` thì cần `d ≥ S/threshold` ⇒ `lo = max(1, ceil(S / threshold))`.

## 7. Các cách tiếp theo

### Cách 2 — Binary search trên đáp án

**Từng đoạn:**
1. `lo, hi = 1, max(nums)`.
2. Lặp `while lo < hi`, tính `mid`, tính `f(mid)` bằng ceil số nguyên.
3. Co khoảng theo quy tắc ở mục 6. Trả `lo`.

In [10]:
def smallest_divisor_bs(nums, threshold):
    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        total = sum((x + mid - 1) // mid for x in nums)
        if total <= threshold:
            hi = mid
        else:
            lo = mid + 1
    return lo

for nums, th, want in examples:
    print(smallest_divisor_bs(nums, th), "| mong đợi", want)

5 | mong đợi 5
44 | mong đợi 44


In [11]:
class SolutionBS:
    def smallestDivisor(self, nums: List[int], threshold: int) -> int:
        lo, hi = 1, max(nums)
        while lo < hi:
            mid = (lo + hi) // 2
            if sum((x + mid - 1) // mid for x in nums) <= threshold:
                hi = mid
            else:
                lo = mid + 1
        return lo

print([SolutionBS().smallestDivisor(n, t) for n, t, _ in examples])

[5, 44]


**Độ phức tạp Cách 2:** `O(n log M)` thời gian (~20 vòng × `n`), `O(1)` bộ nhớ thêm.

### Cách 3 — Dùng `bisect` với `key` (gọn)

Biến `range(1, M + 1)` thành "mảng ảo" các giá trị `False/True` của `ok(d)`, tìm vị trí `True` đầu tiên.

In [12]:
def smallest_divisor_bisect(nums, threshold):
    M = max(nums)
    ok = lambda d: sum(-(-x // d) for x in nums) <= threshold
    return bisect.bisect_left(range(1, M + 1), True, key=ok) + 1

for nums, th, want in examples:
    print(smallest_divisor_bisect(nums, th), "| mong đợi", want)

5 | mong đợi 5
44 | mong đợi 44


`+ 1` vì chỉ số `0` của `range(1, M+1)` tương ứng `d = 1`. Độ phức tạp như Cách 2.

### Cách 4 — Binary search với cận dưới chặt hơn

**Từng đoạn:**
1. `lo = max(1, ceil(S / threshold))` (chứng minh ở mục 6), `hi = max(nums)`.
2. Nếu `lo > hi` thì… không xảy ra, vì `S/threshold ≤ S/n ≤ M`.
3. Binary search như Cách 2 trên khoảng hẹp hơn.

Ngoài ra có thể dùng cận trên `hi = ceil(S / (threshold − n + 1))`… nhưng cẩn thận chứng minh; ở đây giữ `hi = M` cho an toàn.

In [13]:
def smallest_divisor_bs_tight(nums, threshold):
    S = sum(nums)
    lo = max(1, (S + threshold - 1) // threshold)
    hi = max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((x + mid - 1) // mid for x in nums) <= threshold:
            hi = mid
        else:
            lo = mid + 1
    return lo

for nums, th, want in examples:
    S = sum(nums)
    print(smallest_divisor_bs_tight(nums, th), "| mong đợi", want,
          f"| khoảng ban đầu [{max(1, -(-S // th))}, {max(nums)}] thay vì [1, {max(nums)}]")

5 | mong đợi 5 | khoảng ban đầu [3, 9] thay vì [1, 9]
44 | mong đợi 44 | khoảng ban đầu [23, 44] thay vì [1, 44]


**Độ phức tạp Cách 4:** vẫn `O(n log M)` worst, nhưng thường ít vòng hơn.

### Cách 5 (nâng cao) — Tính `f(d)` bằng prefix count

Khi `d` lớn, nhiều phần tử cho cùng giá trị `ceil(x/d) = k` (những `x` thuộc `((k−1)d, kd]`). Với `pre[v]` = số phần tử `≤ v`:

```
f(d) = Σ_{k=1}^{ceil(M/d)} k · (pre[min(kd, M)] − pre[(k−1)d])
```

Chi phí mỗi lần tính là `O(M/d)` thay vì `O(n)`. Kết hợp binary search: `O(M + Σ M/mid)`. Hữu ích khi `n` rất lớn so với `M/d`.

In [14]:
def smallest_divisor_count(nums, threshold):
    M = max(nums)
    pre = [0] * (M + 1)
    for x in nums:
        pre[x] += 1
    for v in range(1, M + 1):
        pre[v] += pre[v - 1]

    def f_fast(d):
        total, k, left = 0, 1, 0
        while left < M:
            right = min(left + d, M)
            total += k * (pre[right] - pre[left])     # các x thuộc (left, right]
            left, k = right, k + 1
        return total

    lo, hi = 1, M
    while lo < hi:
        mid = (lo + hi) // 2
        if f_fast(mid) <= threshold:
            hi = mid
        else:
            lo = mid + 1
    return lo

for nums, th, want in examples:
    print(smallest_divisor_count(nums, th), "| mong đợi", want)

5 | mong đợi 5
44 | mong đợi 44


**Độ phức tạp Cách 5:** `O(n + M)` dựng prefix, mỗi vòng `O(M/mid)`; vòng đầu `mid ≈ M/2` rất rẻ, các vòng sau `mid` nhỏ dần thì đắt dần — worst `O(M)` mỗi vòng ⇒ `O(n + M log M)`. Không phải lúc nào cũng nhanh hơn Cách 2, nhưng là kỹ thuật đáng biết (gặp lại ở các bài "tổng của floor/ceil").

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1** (`lo = 1`, `hi = 9`):

| vòng | lo | hi | mid | f(mid) | ≤ 6? | cập nhật |
|---|---|---|---|---|---|---|
| 1 | 1 | 9 | 5 | 5 | ✔ | hi = 5 |
| 2 | 1 | 5 | 3 | 7 | ✗ | lo = 4 |
| 3 | 4 | 5 | 4 | 7 | ✗ | lo = 5 |
| — | 5 | 5 | | | | dừng → 5 |

In [15]:
def trace_bs(nums, threshold):
    lo, hi = 1, max(nums)
    step = 0
    while lo < hi:
        mid = (lo + hi) // 2
        parts = [(x + mid - 1) // mid for x in nums]
        total = sum(parts)
        step += 1
        ok = total <= threshold
        print(f"vòng {step}: lo={lo}, hi={hi}, mid={mid}, các ceil={parts}, f={total}, "
              f"{'<=' if ok else '>'} {threshold} -> {'hi = mid' if ok else 'lo = mid + 1'}")
        if ok:
            hi = mid
        else:
            lo = mid + 1
    print("đáp án =", lo)
    return lo

trace_bs(*examples[0][:2])
print()
trace_bs(*examples[1][:2])

vòng 1: lo=1, hi=9, mid=5, các ceil=[1, 1, 1, 2], f=5, <= 6 -> hi = mid
vòng 2: lo=1, hi=5, mid=3, các ceil=[1, 1, 2, 3], f=7, > 6 -> lo = mid + 1
vòng 3: lo=4, hi=5, mid=4, các ceil=[1, 1, 2, 3], f=7, > 6 -> lo = mid + 1
đáp án = 5

vòng 1: lo=1, hi=44, mid=22, các ceil=[2, 1, 2, 1, 1], f=7, > 5 -> lo = mid + 1
vòng 2: lo=23, hi=44, mid=33, các ceil=[2, 1, 1, 1, 1], f=6, > 5 -> lo = mid + 1
vòng 3: lo=34, hi=44, mid=39, các ceil=[2, 1, 1, 1, 1], f=6, > 5 -> lo = mid + 1
vòng 4: lo=40, hi=44, mid=42, các ceil=[2, 1, 1, 1, 1], f=6, > 5 -> lo = mid + 1
vòng 5: lo=43, hi=44, mid=43, các ceil=[2, 1, 1, 1, 1], f=6, > 5 -> lo = mid + 1
đáp án = 44


44

## 9. Code nộp LeetCode

Binary search trên đáp án với cận dưới chặt (Cách 4) — `O(n log M)`, chỉ dùng số nguyên.

In [16]:
class Solution:
    def smallestDivisor(self, nums: List[int], threshold: int) -> int:
        lo = max(1, (sum(nums) + threshold - 1) // threshold)   # cận dưới ceil(S/threshold)
        hi = max(nums)                                         # f(max) = n <= threshold
        while lo < hi:
            mid = (lo + hi) // 2
            total = 0
            for x in nums:
                total += (x + mid - 1) // mid                  # ceil(x / mid)
            if total <= threshold:
                hi = mid
            else:
                lo = mid + 1
        return lo

for nums, th, want in examples:
    got = Solution().smallestDivisor(nums, th)
    print(got, "OK" if got == want else "SAI")

5 OK
44 OK


## 10. Vì sao thuật toán đúng

1. **Đơn điệu:** với `d1 < d2`, `ceil(x/d1) ≥ ceil(x/d2)` cho mọi `x > 0` ⇒ `f(d1) ≥ f(d2)`. Vậy nếu `ok(d1)` đúng thì `ok(d2)` đúng.
2. **Tồn tại đáp án trong `[lo, hi]`:** `ok(M)` đúng vì `f(M) = n ≤ threshold`. Mọi `d < ceil(S/threshold)` có `f(d) ≥ S/d > threshold` ⇒ sai; nên đáp án `≥` cận dưới.
3. **Bất biến vòng lặp:** trước mỗi vòng, đáp án `∈ [lo, hi]`. Nếu `ok(mid)` thì đáp án `≤ mid` ⇒ giữ `[lo, mid]`; nếu không thì mọi `d ≤ mid` đều sai (đơn điệu) ⇒ `[mid+1, hi]`.
4. **Kết thúc:** `mid < hi` khi `lo < hi`, nên khoảng co lại thật sự mỗi vòng; dừng khi `lo == hi` = đáp án.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `M = max(nums)`, `S = sum(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n · ans)` ≤ `O(n·M)` | `O(1)` |
| 2. Binary search | `O(n log M)` ≈ `5·10^4 × 20 = 10^6` | `O(1)` |
| 3. bisect + key | `O(n log M)` | `O(1)` |
| 4. BS cận chặt | `O(n log(M − S/threshold))` | `O(1)` |
| 5. BS + prefix count | `O(n + M log M)` worst | `O(M)` |

## 12. Bộ test

Ví dụ đề, edge case (1 phần tử, `threshold = n` buộc `d = max`, `threshold` lớn ⇒ `d = 1`, giá trị cực đại), đối chiếu ngẫu nhiên với brute force (seed cố định, đúng ràng buộc `n ≤ threshold`), và một test lớn đúng cỡ đề.

In [17]:
ALL = {
    "brute": smallest_divisor_brute, "bs": smallest_divisor_bs, "bisect": smallest_divisor_bisect,
    "tight": smallest_divisor_bs_tight, "count": smallest_divisor_count,
    "SolBS": lambda a, t: SolutionBS().smallestDivisor(a, t),
    "Solution": lambda a, t: Solution().smallestDivisor(a, t),
}
fixed = [
    ([1, 2, 5, 9], 6, 5), ([44, 22, 33, 11, 1], 5, 44),
    ([1], 1, 1), ([7], 1, 7), ([7], 7, 1), ([7], 3, 3),
    ([2, 3, 5, 7, 11], 11, 3),
    ([19], 5, 4),
    ([1, 1, 1, 1], 4, 1),
    ([10**6] * 3, 3, 10**6),
    ([10**6], 10**6, 1),
]
for nums, th, want in fixed:
    for name, fn in ALL.items():
        if name == "brute" and want > 10**5:
            continue
        got = fn(nums, th)
        assert got == want, (name, nums, th, got, want)
print("Test cố định: qua", len(fixed), "case")

Test cố định: qua 11 case


In [18]:
rng = random.Random(1283)
for _ in range(1500):
    n = rng.randint(1, 8)
    nums = [rng.randint(1, rng.choice([5, 50, 500])) for _ in range(n)]
    th = rng.randint(n, max(n, sum(nums) + 2))      # đúng ràng buộc n <= threshold
    want = smallest_divisor_brute(nums, th)
    for name, fn in ALL.items():
        assert fn(nums, th) == want, (name, nums, th)
    # kiểm tra định nghĩa: want thoả, want - 1 không thoả
    assert f(nums, want) <= th and (want == 1 or f(nums, want - 1) > th)
print("1500 test ngẫu nhiên: mọi cách khớp brute force và đúng định nghĩa")

1500 test ngẫu nhiên: mọi cách khớp brute force và đúng định nghĩa


In [19]:
big = [rng.randint(1, 10**6) for _ in range(5 * 10**4)]
for th in [5 * 10**4, 10**5, 10**6]:
    t = time.perf_counter()
    a = Solution().smallestDivisor(big, th)
    b = smallest_divisor_bs(big, th)
    assert a == b and f(big, a) <= th and (a == 1 or f(big, a - 1) > th)
    print(f"n=5·10^4, threshold={th}: divisor={a}, {time.perf_counter() - t:.3f}s cho 2 cách")

n=5·10^4, threshold=50000: divisor=999996, 0.102s cho 2 cách


n=5·10^4, threshold=100000: divisor=331820, 0.125s cho 2 cách


n=5·10^4, threshold=1000000: divisor=25476, 0.119s cho 2 cách


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng chia làm tròn xuống `//` thay vì làm tròn lên

In [20]:
def wrong_floor(nums, threshold):
    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(x // mid for x in nums) <= threshold:     # SAI: floor
            hi = mid
        else:
            lo = mid + 1
    return lo

nums, th, want = examples[0]
print("Code sai:", wrong_floor(nums, th), "| đúng là:", want)

Code sai: 3 | đúng là: 5


Với `d = 3`: floor cho `0+0+1+3 = 4 ≤ 6` nên code sai dừng ở 3, trong khi ceil cho `1+1+2+3 = 7 > 6`. **Code đúng:** `(x + mid - 1) // mid`.

### Lỗi 2 — Bắt đầu tìm từ `lo = 0` ⇒ chia cho 0

In [21]:
def wrong_lo_zero(nums, threshold):
    lo, hi = 0, max(nums)                          # SAI: divisor phải >= 1
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((x + mid - 1) // mid for x in nums) <= threshold:
            hi = mid
        else:
            lo = mid + 1
    return lo

try:
    print(wrong_lo_zero([1], 1))
except ZeroDivisionError as e:
    print("Code sai gặp lỗi:", type(e).__name__, "-", e)
print("Code đúng:", Solution().smallestDivisor([1], 1))

Code sai gặp lỗi: ZeroDivisionError - integer division or modulo by zero
Code đúng: 1


Với `nums = [1]`, `hi = 1` ⇒ `mid = 0` ⇒ chia cho 0. **Code đúng:** `lo = 1` (hoặc cận dưới `ceil(S/threshold) ≥ 1`).

### Lỗi 3 — Trộn mẫu `while lo <= hi` với `hi = mid` ⇒ lặp vô hạn

Ta thêm bộ đếm vòng để **dừng an toàn** và in ra thay vì treo notebook.

In [22]:
def wrong_infinite(nums, threshold, max_iter=200):
    lo, hi = 1, max(nums)
    it = 0
    while lo <= hi:                                 # SAI khi đi kèm hi = mid
        it += 1
        if it > max_iter:
            return f"lặp quá {max_iter} vòng (lo={lo}, hi={hi}) -> vô hạn"
        mid = (lo + hi) // 2
        if sum((x + mid - 1) // mid for x in nums) <= threshold:
            hi = mid
        else:
            lo = mid + 1
    return lo

print("Code sai:", wrong_infinite(*examples[0][:2]))
print("Code đúng:", Solution().smallestDivisor(*examples[0][:2]))

Code sai: lặp quá 200 vòng (lo=5, hi=5) -> vô hạn
Code đúng: 5


Khi `lo == hi` và `ok(mid)` đúng, `hi = mid` không đổi gì ⇒ kẹt mãi. **Code đúng:** dùng `while lo < hi` với `hi = mid`, hoặc `while lo <= hi` với `hi = mid - 1` và lưu `ans`.

### Lỗi 4 — Cận trên `hi = max(nums) - 1` (lệch 1)

In [23]:
def wrong_hi(nums, threshold):
    lo, hi = 1, max(nums) - 1                       # SAI: đáp án có thể bằng max
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((x + mid - 1) // mid for x in nums) <= threshold:
            hi = mid
        else:
            lo = mid + 1
    return lo

nums, th, want = examples[1]
print("Code sai:", wrong_hi(nums, th), "| đúng là:", want)

Code sai: 43 | đúng là: 44


Ví dụ 2 có đáp án đúng bằng `max(nums) = 44`. **Code đúng:** `hi = max(nums)`.

### Lỗi 5 — Dùng `math.ceil(x / d)` mà không nghĩ tới số thực

Trong bài này `x ≤ 10^6` nên `math.ceil(x / d)` vẫn đúng, nhưng thói quen này gây sai ở bài có số tới `10^18`.

In [24]:
x, d = 10**18 + 1, 1
print("math.ceil:", math.ceil(x / d), "| đúng:", (x + d - 1) // d, "| bằng nhau?", math.ceil(x / d) == x)

math.ceil: 1000000000000000000 | đúng: 1000000000000000001 | bằng nhau? False


**Code đúng:** luôn dùng `(x + d - 1) // d` hoặc `-(-x // d)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề hỏi "**giá trị nhỏ nhất/lớn nhất sao cho [một đại lượng] ≤/≥ ngưỡng**" và đại lượng **đơn điệu** theo giá trị đó ⇒ **binary search trên đáp án**.
- Viết hàm `ok(x)` riêng, kiểm tra đơn điệu bằng vài giá trị, rồi áp mẫu `first_true`.
- Khoảng tìm kiếm: tìm một giá trị **chắc chắn sai** và một giá trị **chắc chắn đúng** (ở đây `hi = max(nums)` chắc chắn đúng).
- Ceil số nguyên: `(a + b - 1) // b`. Học thuộc.
- Ràng buộc `n·log M ≈ 10^6` ⇒ binary search thoải mái trong Python.

## 15. Biến thể của bài

1. **LC 875 Koko Eating Bananas:** tốc độ ăn nhỏ nhất để ăn hết trong `h` giờ — **giống hệt**: `Σ ceil(piles[i]/k) ≤ h`.
2. **LC 1011 Capacity To Ship Packages Within D Days:** sức chứa nhỏ nhất ⇒ `ok(cap)` = số ngày cần (tham lam) `≤ D`.
3. **LC 410 Split Array Largest Sum:** tương tự 1011.
4. **LC 1482 Minimum Number of Days to Make m Bouquets:** binary search trên số ngày.
5. **LC 2187 Minimum Time to Complete Trips:** `Σ floor(t / time[i]) ≥ totalTrips`, tìm `t` nhỏ nhất.
6. **Tìm divisor lớn nhất sao cho tổng ≥ threshold** ⇒ đảo chiều vị từ (`last_true`).

In [25]:
def min_eating_speed(piles, h):            # LC 875 — cùng một khuôn
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((p + mid - 1) // mid for p in piles) <= h:
            hi = mid
        else:
            lo = mid + 1
    return lo

print(min_eating_speed([3, 6, 7, 11], 8), min_eating_speed([30, 11, 23, 4, 20], 5),
      min_eating_speed([30, 11, 23, 4, 20], 6))   # 4, 30, 23 theo đề LC 875

4 30 23


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử `d = 1, 2, …` | `O(n·M)` | `O(1)` | kiểm chứng, dữ liệu nhỏ |
| 2. Binary search | `f` đơn điệu ⇒ tìm `True` đầu tiên | `O(n log M)` | `O(1)` | chuẩn, dễ nhớ |
| 3. `bisect` + `key` | như 2, dùng thư viện | `O(n log M)` | `O(1)` | code ngắn (Python ≥ 3.10) |
| 4. BS cận chặt | `lo = ceil(S/threshold)` | `O(n log M)` | `O(1)` | **nộp bài**, ít vòng hơn |
| 5. BS + prefix count | gom các `x` cùng giá trị ceil | `O(n + M log M)` | `O(M)` | khi `n` rất lớn, `M` nhỏ |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `smallestDivisor([2, 3, 5, 7, 11], 11)` và `smallestDivisor([19], 5)` bằng bao nhiêu?

**Bài 2 (tính tay):** Với `nums = [21212, 10101, 12121]`, `threshold = 1000000`, đáp án là gì? Vì sao không cần tính?

**Bài 3 (điền TODO):** Hoàn thiện `last_true_todo(lo, hi, ok)` tìm `x` **lớn nhất** trong `[lo, hi]` có `ok(x)` đúng, biết `ok` có dạng `True…True False…False`.

**Bài 4 (tự viết lại):** Viết `ship_within_days(weights, days)` (LC 1011): sức chứa nhỏ nhất của tàu để chở hết các kiện **theo thứ tự** trong `days` ngày.

**Bài 5 (mini challenge):** Viết `largest_divisor_at_least(nums, target)`: tìm `d` **lớn nhất** sao cho `Σ ceil(nums[i]/d) ≥ target` (giả sử `target ≤ sum(nums)` để luôn có đáp án).

In [26]:
def last_true_todo(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi + 1) // 2     # chú ý +1 để tránh lặp vô hạn
        # TODO: nếu ok(mid) thì lo = ?, ngược lại hi = ?
        pass
    return lo

def ship_within_days_todo(weights, days):
    # TODO: lo = max(weights), hi = sum(weights); ok(cap) = số ngày tham lam <= days
    pass

# Lưu ý: cell này chỉ định nghĩa hàm, không gọi.

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Lập bảng `f(d)` với `d` nhỏ. (2) `[2,3,5,7,11]`: `f(2) = 16`, `f(3) = 11`. (3) `[19]`: cần `ceil(19/d) ≤ 5` ⇒ `d ≥ 19/5 = 3.8`.

**Bài 2.** (1) `threshold` rất lớn so với tổng. (2) `f(1) = S = 43434 ≤ 10^6`. (3) Vậy đáp án là `1`.

**Bài 3.** (1) `ok(mid)` đúng ⇒ đáp án `≥ mid` ⇒ `lo = mid`. (2) Sai ⇒ `hi = mid - 1`. (3) Dùng `mid = (lo + hi + 1) // 2` để `mid > lo`, tránh kẹt.

**Bài 4.** (1) Sức chứa phải `≥ max(weights)`. (2) `ok(cap)`: duyệt, cộng dồn, vượt `cap` thì sang ngày mới. (3) Tìm `cap` nhỏ nhất có số ngày `≤ days`.

**Bài 5.** (1) `f(d)` không tăng ⇒ `f(d) ≥ target` có dạng `True…False`. (2) Dùng `last_true` trên `[1, max(nums)]`. (3) `f(1) = S ≥ target` nên luôn có đáp án.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert Solution().smallestDivisor([2, 3, 5, 7, 11], 11) == 3
assert Solution().smallestDivisor([19], 5) == 4
# Bài 2
assert Solution().smallestDivisor([21212, 10101, 12121], 1000000) == 1

# Bài 3
def last_true_sol(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

assert last_true_sol(0, 100, lambda x: x * x <= 50) == 7

# Bài 4
def ship_within_days_sol(weights, days):
    def need_days(cap):
        d, cur = 1, 0
        for w in weights:
            if cur + w > cap:
                d, cur = d + 1, 0
            cur += w
        return d
    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if need_days(mid) <= days:
            hi = mid
        else:
            lo = mid + 1
    return lo

assert ship_within_days_sol([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5) == 15   # ví dụ LC 1011
assert ship_within_days_sol([3, 2, 2, 4, 1, 4], 3) == 6
assert ship_within_days_sol([1, 2, 3, 1, 1], 4) == 3

# Bài 5
def largest_divisor_at_least_sol(nums, target):
    return last_true_sol(1, max(nums), lambda d: f(nums, d) >= target)

def days_needed_brute(weights, cap):
    """Thử mọi cách đặt vạch chia ngày (bitmask) và lấy số ngày ít nhất mà mỗi ngày <= cap."""
    n = len(weights)
    best = n + 1
    for mask in range(1 << (n - 1)):
        days_used, cur, fine = 1, 0, True
        for i, x in enumerate(weights):
            cur += x
            if cur > cap:
                fine = False
                break
            if i < n - 1 and mask >> i & 1:
                days_used, cur = days_used + 1, 0
        if fine:
            best = min(best, days_used)
    return best

rng = random.Random(42)
for _ in range(500):
    nums = [rng.randint(1, 60) for _ in range(rng.randint(1, 6))]
    target = rng.randint(1, sum(nums))
    want = max(d for d in range(1, max(nums) + 1) if f(nums, d) >= target)
    assert largest_divisor_at_least_sol(nums, target) == want
    w = [rng.randint(1, 20) for _ in range(rng.randint(1, 7))]
    days = rng.randint(1, len(w))
    brute_cap = next(c for c in range(max(w), sum(w) + 1) if days_needed_brute(w, c) <= days)
    assert ship_within_days_sol(w, days) == brute_cap
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template binary search trên đáp án (tìm nhỏ nhất thoả):**

```python
def ok(x): ...                  # đơn điệu: False...False True...True
lo, hi = <chắc chắn ≤ đáp án>, <chắc chắn ok>
while lo < hi:
    mid = (lo + hi) // 2
    if ok(mid): hi = mid
    else:       lo = mid + 1
return lo
```

**Tìm lớn nhất thoả** (`True…True False…False`): `mid = (lo + hi + 1) // 2`, `ok ⇒ lo = mid`, ngược lại `hi = mid - 1`.

**Ceil số nguyên:** `(a + b - 1) // b` hoặc `-(-a // b)`.

**Checklist trước khi submit:**
- [ ] Ceil, không phải floor.
- [ ] `lo ≥ 1` (không chia cho 0).
- [ ] `hi = max(nums)` (đáp án có thể bằng max).
- [ ] `while lo < hi` đi với `hi = mid`; không trộn với `lo <= hi`.
- [ ] Kiểm tra lại `f(ans) ≤ threshold` và `f(ans − 1) > threshold` trên vài test.
- [ ] Đúng chữ ký: `smallestDivisor(self, nums: List[int], threshold: int) -> int`.